# Imports

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from uavloc.builds import build_dir
from uavloc.data.annotated import IGNORE_LABEL
from uavloc.features.imagenet import to_batch
from uavloc.features.segmentation import CLASSES
from uavloc.plotting.correlation import SEG_PALETTE
from uavloc.segmentation.boundary import label_band
from uavloc.segmentation.eval import (MEAN_KEY, VAL_SETS, bilinear_argmax, evaluate, evaluate_predictions,
                                      label_at_stride, load_baseline_model, load_val_sets, lraspp_outputs,
                                      nearest_argmax, results_table, run_model, upsample_logits)
from uavloc.segmentation.latency import model_latency, refiner_latency, standard_tile
from uavloc.segmentation.metrics import segmentation_scores
from uavloc.track import REPO_ROOT

# Params

In [ ]:
SLUG = "seg-boundary-eval"
DEVICE = "cpu"
BOUNDARY_PX = 4
BF_TOLERANCES_PX = [1, 2]
D_SWEEP_PX = [1, 2, 3, 4, 6, 8]
VILLAGE_SCENE = {"size_px": 512, "track_px": 12, "building_px": 50}
BLOCK_SCENE = {"size_px": 1024, "track_px": 64, "building_px": 200}
SYNTH_SHIFTS_PX = [0, 1, 2, 3, 4, 6, 8, 12]
SYNTH_IGNORE_COLS = 24
SYNTH_SEED = 0
FLAT_SHIFT_PX = 2
FLAT_MIOU_MAX_DROP = 0.05
BIOU_MIN_DROP = 0.2
PROBE_RESULTS = "pocs/mobilenet-lraspp-speed/results-lraspp.json"
PROBE_MIOU_TOLERANCE = 0.002
LATENCY_SET = "bushcha"
LATENCY_TOP_LEFT_PX = [1024, 1024]
LATENCY_THREADS = 4
LATENCY_WARMUP = 5
LATENCY_RUNS = 30
ZOOM_SET = "birky-strip-google"
ZOOM_BOX_PX = [120, 1400, 376, 1912]

# Data

In [ ]:
FIGURES = build_dir(SLUG, create=True) / "figures"
FIGURES.mkdir(exist_ok=True)
TOLERANCES = tuple(BF_TOLERANCES_PX)
torch.manual_seed(SYNTH_SEED)

In [ ]:
val_sets = load_val_sets()
pd.DataFrame([{"set": n, "date": a.date, "size_px": a.label.shape, "labelled": (a.label != IGNORE_LABEL).mean(),
               **{c: (a.label == k).mean() for k, c in enumerate(CLASSES)}}
              for n, items in val_sets.items() for a in items]).round(3)

In [ ]:
def synthetic_label(size_px: int, track_px: int, building_px: int, ignore_cols: int) -> np.ndarray:
    """A wood, a pond, a crossing track and two buildings, scaled to `size_px`; the right edge ignored."""
    n, half = size_px, track_px // 2
    lab = np.zeros((n, n), np.uint8)
    yy, xx = np.mgrid[:n, :n]
    lab[(yy - 0.7 * n) ** 2 + (xx - 0.3 * n) ** 2 < (0.2 * n) ** 2] = CLASSES.index("woody")
    lab[(yy - 0.25 * n) ** 2 + (xx - 0.75 * n) ** 2 < (0.08 * n) ** 2] = CLASSES.index("water")
    lab[:, n // 2 - half:n // 2 + half] = lab[n // 3 - half:n // 3 + half, :] = CLASSES.index("track")
    for top, left in [(int(0.08 * n), int(0.12 * n)), (int(0.72 * n), int(0.6 * n))]:
        lab[top:top + building_px, left:left + building_px] = CLASSES.index("building")
    lab[:, n - ignore_cols:] = IGNORE_LABEL
    return lab

In [ ]:
scenes = {"village": synthetic_label(**VILLAGE_SCENE, ignore_cols=SYNTH_IGNORE_COLS),
          "blocks": synthetic_label(**BLOCK_SCENE, ignore_cols=SYNTH_IGNORE_COLS)}
fig, axes = plt.subplots(1, len(scenes), figsize=(8, 4))
for ax, (name, lab) in zip(axes, scenes.items()):
    ax.imshow(lab, cmap="tab10", interpolation="nearest"); ax.set_title(name); ax.axis("off")

# Model

In [ ]:
model = load_baseline_model(DEVICE)

In [ ]:
check_tile = standard_tile(val_sets[LATENCY_SET][0].rgb, *LATENCY_TOP_LEFT_PX)
check_out = lraspp_outputs(model, check_tile, DEVICE)
with torch.no_grad():
    forward_probs = model(to_batch(check_tile, DEVICE)).softmax(1)[0].cpu().numpy()
{"logits_s8": check_out.logits.shape, "probs": check_out.probs.shape,
 "max |forward - coarse+upsample|": float(np.abs(forward_probs - check_out.probs).max())}

# Run

In [ ]:
def shifted(label: np.ndarray, shift_px: int) -> np.ndarray:
    """The label moved `shift_px` right and down, as a prediction (ignored pixels become background)."""
    pred = np.roll(label, (shift_px, shift_px), axis=(0, 1)).copy()
    pred[pred == IGNORE_LABEL] = 0
    return pred

In [ ]:
def scalar_scores(pred: np.ndarray, label: np.ndarray, boundary_px: int) -> dict:
    scores = segmentation_scores([pred], [label], boundary_px, TOLERANCES)
    return {k: v for k, v in scores.items() if not isinstance(v, dict)}

In [ ]:
def shift_curve(label: np.ndarray) -> pd.DataFrame:
    curve = pd.DataFrame({k: scalar_scores(shifted(label, k), label, BOUNDARY_PX) for k in SYNTH_SHIFTS_PX}).T
    curve.index.name = "shift_px"
    return curve

In [ ]:
curves = {name: shift_curve(lab) for name, lab in scenes.items()}
pd.concat(curves, axis=1).round(3)

In [ ]:
def check_curve(curve: pd.DataFrame, monotone_up_to_px: int) -> None:
    """Self-score 1; strictly falling Boundary IoU and non-increasing everything up to `monotone_up_to_px`."""
    assert np.allclose(curve.loc[0], 1.0), "a mask against itself must score 1"
    head = curve.loc[:monotone_up_to_px]
    assert (head.diff().dropna() <= 1e-12).all().all(), head
    assert (head["biou"].diff().dropna() < 0).all(), head["biou"]

In [ ]:
check_curve(curves["village"], BOUNDARY_PX)
check_curve(curves["blocks"], max(SYNTH_SHIFTS_PX))
drops = (1 - curves["blocks"].loc[FLAT_SHIFT_PX])
assert drops["miou"] < FLAT_MIOU_MAX_DROP and drops["biou"] > BIOU_MIN_DROP, drops
drops.round(3)

In [ ]:
synth = scenes["village"]
noisy_ignore = synth.copy()
ignored = synth == IGNORE_LABEL
noisy_ignore[ignored] = np.random.default_rng(SYNTH_SEED).integers(len(CLASSES), size=ignored.sum())
ignore_check = scalar_scores(noisy_ignore, synth, BOUNDARY_PX)
assert np.allclose(list(ignore_check.values()), 1.0), ignore_check
ignore_check

In [ ]:
cases = run_model(model, val_sets, DEVICE)

In [ ]:
probe_final = json.loads((REPO_ROOT / PROBE_RESULTS).read_text())["final_scores"]
baseline = evaluate(bilinear_argmax, cases, BOUNDARY_PX, TOLERANCES)
probe_check = pd.DataFrame({"probe.ipynb": {n: probe_final[n]["miou"] for n in VAL_SETS},
                            "this build": {n: baseline[n]["miou"] for n in VAL_SETS}})
assert (probe_check.diff(axis=1).iloc[:, 1].abs() < PROBE_MIOU_TOLERANCE).all(), probe_check
probe_check.round(4)

In [ ]:
def oracle_preds(cases: dict, mode: str) -> dict[str, list[np.ndarray]]:
    return {n: [label_at_stride(c.image.label, mode) for c in items] for n, items in cases.items()}

In [ ]:
runs = {
    "LR-ASPP bilinear (baseline)": baseline,
    "LR-ASPP stride-8 nearest": evaluate(nearest_argmax, cases, BOUNDARY_PX, TOLERANCES),
    "oracle: label at stride 8, nearest": evaluate_predictions(oracle_preds(cases, "nearest"), cases,
                                                               BOUNDARY_PX, TOLERANCES),
    "oracle: label at stride 8, bilinear": evaluate_predictions(oracle_preds(cases, "bilinear"), cases,
                                                                BOUNDARY_PX, TOLERANCES),
}

In [ ]:
summary = pd.concat({name: results_table(r) for name, r in runs.items()}, names=["prediction", "set"])
summary.round(3)

In [ ]:
per_class = pd.concat({m: pd.DataFrame({n: baseline[n][f"{m}_per_class"] for n in VAL_SETS}).T
                       for m in ["iou", "biou", "bf1_1px", "bf1_2px"]}, axis=1)
per_class.round(3)

In [ ]:
def d_sweep(preds: dict[str, list[np.ndarray]]) -> dict[int, float]:
    return {d: evaluate_predictions(preds, cases, d, TOLERANCES)[MEAN_KEY]["biou"] for d in D_SWEEP_PX}

In [ ]:
run_preds = {"LR-ASPP bilinear (baseline)": {n: [bilinear_argmax(c.output, c.image.rgb) for c in items]
                                             for n, items in cases.items()},
             "LR-ASPP stride-8 nearest": {n: [nearest_argmax(c.output, c.image.rgb) for c in items]
                                          for n, items in cases.items()},
             "oracle: label at stride 8, nearest": oracle_preds(cases, "nearest")}
sweep = pd.DataFrame({name: d_sweep(p) for name, p in run_preds.items()})
sweep.index.name = "boundary_px"
sweep.round(3)

In [ ]:
latency_tile = standard_tile(val_sets[LATENCY_SET][0].rgb, *LATENCY_TOP_LEFT_PX)
latency_output = lraspp_outputs(model, latency_tile, DEVICE)
timing = dict(warmup=LATENCY_WARMUP, runs=LATENCY_RUNS, threads=LATENCY_THREADS)
latency = {"model (normalise -> stride-8 logits -> bilinear -> softmax)": model_latency(model, latency_tile, **timing),
           "refine: argmax of bilinear probs": refiner_latency(bilinear_argmax, latency_output, latency_tile, **timing),
           "refine: stride-8 nearest argmax": refiner_latency(nearest_argmax, latency_output, latency_tile, **timing)}
latency_table = pd.DataFrame({k: vars(v) for k, v in latency.items()}).T
latency_table.round(1)

In [ ]:
results = {"boundary_px": BOUNDARY_PX, "bf_tolerances_px": list(TOLERANCES),
           "runs": {name: r for name, r in runs.items()}, "biou_d_sweep": sweep.to_dict(),
           "latency_512px": {k: vars(v) for k, v in latency.items()},
           "latency_tile": {"set": LATENCY_SET, "date": val_sets[LATENCY_SET][0].date, "top_left_px": LATENCY_TOP_LEFT_PX},
           "probe_check": probe_check.to_dict()}
(build_dir(SLUG) / "results.json").write_text(json.dumps(results, indent=1, default=str))

# Plots

In [ ]:
def colour(lab: np.ndarray) -> np.ndarray:
    out = np.full((*lab.shape, 3), 255, np.uint8)
    ok = lab != IGNORE_LABEL
    out[ok] = SEG_PALETTE[lab[ok]]
    return out

In [ ]:
fig, axes = plt.subplots(1, len(curves), figsize=(13, 4), sharey=True)
for ax, (name, curve) in zip(axes, curves.items()):
    for col, style in zip(curve.columns, ["-o", "-s", "-^", "-v"]):
        ax.plot(curve.index, curve[col], style, label=col)
    ax.set_xlabel("prediction shifted by k px (right and down)"); ax.set_title(f"{name} scene")
    ax.grid(alpha=.3); ax.legend()
axes[0].set_ylabel(f"score (Boundary IoU d = {BOUNDARY_PX} px)")
fig.savefig(FIGURES / "synthetic-shift.png", dpi=120, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name in sweep.columns:
    ax.plot(sweep.index, sweep[name], "-o", label=name)
ax.axvline(BOUNDARY_PX, color="0.5", ls="--", lw=1)
ax.set_xlabel("Boundary IoU dilation d, px (0.385 m/px)"); ax.set_ylabel("mean Boundary IoU over val sets")
ax.set_title("Boundary IoU vs d"); ax.grid(alpha=.3); ax.legend(fontsize=8)
fig.savefig(FIGURES / "biou-vs-d.png", dpi=120, bbox_inches="tight")

In [ ]:
def band_overlay(pred: np.ndarray, label: np.ndarray) -> np.ndarray:
    """Predicted classes, with the label's 1-px class contours drawn in white."""
    out = colour(pred)
    edges = np.any([label_band(label, c, 1) for c in range(len(CLASSES))], axis=0)
    out[edges] = 255
    return out

In [ ]:
top, left, bottom, right = ZOOM_BOX_PX
zoom_case = cases[ZOOM_SET][0]
zoom = np.s_[top:bottom, left:right]
lab = zoom_case.image.label[zoom]
panels = [(zoom_case.image.rgb[zoom], f"{ZOOM_SET} RGB"), (colour(lab), "label"),
          (band_overlay(bilinear_argmax(zoom_case.output, zoom_case.image.rgb)[zoom], lab), "LR-ASPP bilinear"),
          (band_overlay(nearest_argmax(zoom_case.output, zoom_case.image.rgb)[zoom], lab), "LR-ASPP stride-8 nearest"),
          (band_overlay(label_at_stride(zoom_case.image.label, "nearest")[zoom], lab), "oracle: label at stride 8")]
fig, axes = plt.subplots(len(panels), 1, figsize=(10, 3.4 * len(panels)))
for ax, (img, title) in zip(axes, panels):
    ax.imshow(img, interpolation="nearest"); ax.set_title(title); ax.axis("off")
fig.legend([plt.Rectangle((0, 0), 1, 1, color=SEG_PALETTE[c] / 255) for c in range(len(CLASSES))], CLASSES,
           loc="lower center", ncol=len(CLASSES))
plt.tight_layout(rect=(0, .02, 1, 1)); fig.savefig(FIGURES / "zoom-baseline.png", dpi=100, bbox_inches="tight")